# Quantisasi Baseline DONUT (PTQ) — `DONUT-Base-Q`

Notebook mandiri untuk menghasilkan model **`DONUT-Base-Q`**: model baseline
`naver-clova-ix/donut-base-finetuned-cord-v2` yang dikuantisasi secara
**weight-only INT8 pada decoder** (Post-Training Quantization via torchao),
dievaluasi dengan metrik yang sama persis dengan pipeline eksperimen utama
(`thesis_v3.ipynb`), lalu disimpan sebagai artifact MLflow di Databricks.

Metrik yang dievaluasi: **Field-Level Precision, Recall, F1-Score, N-TED,
ukuran model (MB), latensi inferensi (ms/sample), dan FLOPs (GFLOPs)** — beserta
kolom perbandingan terhadap baseline (`Size/Latency/FLOPs Reduction (%)`,
`F1 Drop`, `N-TED Increase`, `PTQ Size/Latency Reduction (%)`).

Alur notebook:

1. **Setup & Konfigurasi** — instalasi dependency, reproducibility, dan koneksi MLflow Databricks.
2. **Loading Dataset & Model Baseline** — memuat split CORD-v2, `DonutProcessor`, dan model baseline.
3. **Evaluasi & Metrik** — seluruh fungsi evaluasi (F1/N-TED, ukuran, latensi, FLOPs) identik dengan `thesis_v3.ipynb`.
4. **Weight-Only Quantization (PTQ)** — `quantize_` decoder dengan `Int8WeightOnlyConfig`, verifikasi layer terkuantisasi, lalu evaluasi ulang `DONUT-Base-Q`.
5. **Logging ke Databricks (MLflow)** — run `DONUT-Base` dan `DONUT-Base-Q` lengkap dengan parameter, metrik, tag, artifact laporan (`result.json`, `environment.json`, `requirements.txt`), dan bobot model.
6. **Tabel hasil akhir** — perbandingan dua model beserta kolom reduksi, disimpan sebagai `hasil_eksperimen_donut_quantisasi.json/.csv`.

> **Catatan:** notebook ini harus dijalankan pada **GPU** karena pengukuran latensi
> memakai `torch.cuda.synchronize()`. Kuantisasi bersifat decoder-only (aktivasi tetap
> FP32), konsisten dengan ruang lingkup penelitian.


In [ ]:
!pip install -q "mlflow>=3, <4" databricks-sdk transformers datasets sentencepiece accelerate zss fvcore "torchao>=0.17,<0.19"

In [ ]:
import os
import sys
import re
import json
import time
import copy
import random
import platform
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchao.quantization import Int8WeightOnlyConfig, quantize_

import transformers
from transformers import DonutProcessor, VisionEncoderDecoderModel
from datasets import load_dataset

import mlflow
from mlflow.tracking import MlflowClient
from google.colab import userdata

from PIL import Image
from zss import Node, simple_distance
from fvcore.nn import FlopCountAnalysis

import warnings
warnings.filterwarnings("ignore")

## 1 Setup & Konfigurasi

### 1.1 Konfigurasi Reproducibility

In [ ]:
# Reproducibility Config
SEED = 27
CPU_NUM_THREADS = 4
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
def _set_global_seed(seed=SEED):
    """
    Mengunci seluruh sumber randomness ke satu nilai seed agar eksperimen reproducible.

    Mencakup seed Python (`random`, `PYTHONHASHSEED`), NumPy, PyTorch (CPU dan CUDA),
    serta randomness internal HuggingFace via `transformers.set_seed`. Mode
    deterministik cuDNN/PyTorch juga diaktifkan supaya perbandingan antar teknik
    kompresi tidak terkontaminasi variasi acak training.

    Args:
        seed: nilai seed; default `SEED` global.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    transformers.set_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)


_set_global_seed(SEED)
torch.set_num_threads(CPU_NUM_THREADS)

### 1.2 Koneksi MLflow Databricks

In [ ]:
# MLFlow DataBricks Config
MLFLOW_EXPERIMENT = "/Shared/DONUT-CORD-v2-Optimization"
EXPERIMENT_GROUP = f"donut-cord-v2-seed-{SEED}"


def _setup_mlflow_databricks():
    """
    Menyiapkan koneksi MLflow ke tracking server Databricks.

    Kredensial diambil dari Colab Secrets (`DATABRICKS_HOST`, `DATABRICKS_TOKEN`),
    host dinormalisasi, lalu MLflow client diarahkan ke backend Databricks dan
    experiment diverifikasi lewat `MlflowClient.get_experiment`. Kredensial atau
    path experiment yang salah gagal cepat di sini, alih-alih di tengah proses
    training yang panjang.

    Returns:
        Objek experiment MLflow yang aktif.
    """

    try:
        databricks_host = userdata.get("DATABRICKS_HOST")
        databricks_token = userdata.get("DATABRICKS_TOKEN")
    except Exception as e:
        raise RuntimeError(
            "Secrets belum di setup untuk DATABRICKS_HOST dan DATABRICKS_TOKEN"
        ) from e

    databricks_host = databricks_host.strip().rstrip("/")

    if not databricks_host.startswith(("http://", "https://")):
        databricks_host = "https://" + databricks_host

    os.environ["DATABRICKS_HOST"] = databricks_host
    os.environ["DATABRICKS_TOKEN"] = databricks_token

    mlflow.set_tracking_uri("databricks")

    experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT)

    client = MlflowClient()
    client.get_experiment(experiment.experiment_id)

    return experiment


MLFLOW_EXPERIMENT_INFO = _setup_mlflow_databricks()

## 2 Konfigurasi Global & Loading Baseline

In [ ]:
MODEL_NAME = "naver-clova-ix/donut-base-finetuned-cord-v2"
DATASET_NAME = "naver-clova-ix/cord-v2"

MAX_LENGTH = 512

EVAL_SAMPLES = 100
LATENCY_SAMPLES = 50
WARMUP_SAMPLES = 20

TRAIN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_DEVICE = "cuda"

print("Train device:", TRAIN_DEVICE)
print("Eval device :", EVAL_DEVICE)


In [ ]:
dataset = load_dataset(DATASET_NAME)

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

processor = DonutProcessor.from_pretrained(MODEL_NAME)
baseline_model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)


In [ ]:
def setup_donut_config(model, processor):
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.eos_token_id
    model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(
        "<s_cord-v2>"
    )
    model.config.vocab_size = model.config.decoder.vocab_size
    return model


baseline_model = setup_donut_config(baseline_model, processor)
baseline_model.to(TRAIN_DEVICE)

print("Train data:", len(train_data))
print("Validation data:", len(val_data))
print("Test data:", len(test_data))


## 3 Evaluasi & Metrik

In [ ]:
def clean_sequence(sequence):
    """
    Membersihkan token khusus dari hasil decode sebelum diparsing menjadi JSON.

    `batch_decode` sengaja dipanggil dengan `skip_special_tokens=False` supaya
    tag struktur `<s_key>`/`</s_key>` tetap ada. Konsekuensinya token EOS dan
    PAD ikut terbawa dan harus dibuang manual di sini.

    Args:
        sequence: string hasil `processor.batch_decode`.

    Returns:
        str tanpa token EOS/PAD dan tanpa spasi di ujung.
    """
    sequence = sequence.replace(processor.tokenizer.eos_token, "")
    sequence = sequence.replace(processor.tokenizer.pad_token, "")
    sequence = sequence.strip()
    return sequence

In [ ]:
def predict_json(model, image, device="cpu"):
    """
    Menjalankan inferensi satu gambar dokumen dan mengembalikan hasilnya sebagai JSON.

    Decoding dibuat deterministik (`do_sample=False`, `num_beams=1` alias greedy)
    supaya perbandingan antar model tidak terkontaminasi variasi sampling: model
    yang sama selalu menghasilkan output yang sama untuk gambar yang sama.

    Bila sequence yang dihasilkan rusak secara struktur (tag tidak berpasangan),
    `token2json` gagal dan fungsi mengembalikan dict kosong. Prediksi tersebut
    diperlakukan sebagai jawaban yang seluruh field-nya salah, bukan sebagai
    error yang menghentikan proses evaluasi.

    Args:
        model: model Donut yang dievaluasi.
        image: PIL Image dokumen.
        device: device inferensi ("cuda"/"cpu"). Model terkuantisasi dievaluasi di CUDA.

    Returns:
        tuple `(pred_json, sequence)` berisi hasil parsing dan string mentahnya.
    """
    model.eval()
    model.to(device)

    image = image.convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    decoder_input_ids = processor.tokenizer(
        "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(device)

    with torch.no_grad():
        output_ids = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=MAX_LENGTH,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            # deterministic decoding
            do_sample=False,
            num_beams=1,
        )

    sequence = processor.batch_decode(output_ids, skip_special_tokens=False)[0]
    sequence = clean_sequence(sequence)

    try:
        pred_json = processor.token2json(sequence)
    except:
        pred_json = {}

    return pred_json, sequence

In [ ]:
def normalize_text(text):
    """
    Menormalkan nilai field sebelum dibandingkan dengan ground truth.

    Lowercase dan perapian whitespace supaya perbedaan kapitalisasi atau spasi
    ganda tidak dihitung sebagai kesalahan ekstraksi.

    Args:
        text: nilai apa pun; dikonversi ke str lebih dulu.

    Returns:
        str hasil normalisasi.
    """
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
def flatten_json(obj, prefix=""):
    """
    Meratakan JSON bersarang menjadi dict datar berkunci path.

    Perbandingan field-level membutuhkan unit yang bisa dicocokkan satu per satu.
    Struktur bersarang diratakan menjadi bentuk seperti `menu.0.nm` -> "es teh",
    sehingga kesamaan prediksi dan ground truth dapat dihitung sebagai irisan
    himpunan pasangan (path, nilai).

    Args:
        obj: dict, list, atau skalar yang akan diratakan.
        prefix: path induk; dipakai internal saat rekursi.

    Returns:
        dict {path: nilai ternormalisasi}.
    """
    items = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_key = f"{prefix}.{key}" if prefix else key
            items.update(flatten_json(value, new_key))

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            new_key = f"{prefix}.{i}"
            items.update(flatten_json(value, new_key))

    else:
        items[prefix] = normalize_text(obj)

    return items

In [ ]:
def field_level_f1(pred_json, true_json):
    """
    Menghitung Precision, Recall, dan F1 pada level pasangan (field, nilai).

    Prediksi dan ground truth diratakan lalu diperlakukan sebagai himpunan:
    true positive adalah pasangan yang path sekaligus nilainya sama persis.
    Metrik ini bersifat exact-match per field, sehingga nilai yang benar tetapi
    ditempatkan pada path yang salah dihitung sebagai false positive sekaligus
    false negative.

    Konstanta 1e-8 pada penyebut mencegah pembagian nol ketika model gagal
    menghasilkan JSON dan prediksinya kosong.

    Args:
        pred_json: hasil ekstraksi model.
        true_json: `gt_parse` ground truth.

    Returns:
        tuple `(precision, recall, f1)`.
    """
    pred_items = set(flatten_json(pred_json).items())
    true_items = set(flatten_json(true_json).items())

    tp = len(pred_items & true_items)
    fp = len(pred_items - true_items)
    fn = len(true_items - pred_items)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

In [ ]:
def json_to_tree(obj, name="root"):
    """
    Mengubah JSON menjadi pohon `zss.Node` untuk perhitungan Tree Edit Distance.

    F1 field-level tidak melihat struktur: prediksi dengan seluruh field benar
    tetapi hierarki salah tetap memperoleh nilai tinggi. Representasi pohon
    dipakai agar kesalahan struktur ikut terukur — key menjadi node internal,
    nilai menjadi node daun, dan elemen list diberi label posisi `item_i`.

    Args:
        obj: dict, list, atau skalar.
        name: label node saat ini.

    Returns:
        `zss.Node` akar dari subtree.
    """
    node = Node(str(name))

    if isinstance(obj, dict):
        for key, value in obj.items():
            child = json_to_tree(value, key)
            node.addkid(child)

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            child = json_to_tree(value, f"item_{i}")
            node.addkid(child)

    else:
        value_node = Node(normalize_text(obj))
        node.addkid(value_node)

    return node

In [ ]:
def count_nodes(node):
    """
    Menghitung total node dalam sebuah pohon secara rekursif.

    Dipakai sebagai normalizer N-TED agar jarak edit tidak bias terhadap ukuran
    dokumen: struk yang panjang otomatis punya jarak edit lebih besar meskipun
    kualitas ekstraksinya setara.

    Args:
        node: `zss.Node` akar.

    Returns:
        int jumlah node termasuk akar.
    """
    total = 1
    for child in node.children:
        total += count_nodes(child)
    return total

In [ ]:
def normalized_tree_edit_distance(pred_json, true_json):
    """
    Menghitung Normalized Tree Edit Distance (N-TED) antara prediksi dan ground truth.

    TED adalah jumlah minimum operasi (insert, delete, rename node) yang
    dibutuhkan untuk mengubah pohon prediksi menjadi pohon ground truth;
    `label_distance` memberi biaya 1 untuk label berbeda dan 0 untuk label sama.
    Hasilnya dibagi jumlah node pohon terbesar supaya sebanding antar dokumen
    dengan ukuran berbeda.

    Arah metrik ini berlawanan dengan F1: **semakin kecil semakin baik**, 0
    berarti struktur prediksi identik dengan ground truth.

    Args:
        pred_json: hasil ekstraksi model.
        true_json: `gt_parse` ground truth.

    Returns:
        float N-TED.
    """
    pred_tree = json_to_tree(pred_json)
    true_tree = json_to_tree(true_json)

    def label_distance(a, b):
        return 0 if a == b else 1

    distance = simple_distance(pred_tree, true_tree, label_dist=label_distance)

    normalizer = max(count_nodes(pred_tree), count_nodes(true_tree), 1)
    nted = distance / normalizer

    return nted

In [ ]:
def evaluate_extraction(model, data, device="cpu", max_samples=100):
    """
    Mengukur kualitas ekstraksi model pada sejumlah sample test.

    Prediksi dijalankan satu per satu (batch size 1) karena `generate`
    menghasilkan sequence dengan panjang berbeda-beda tiap dokumen. Skor tiap
    sample dirata-rata secara makro: setiap dokumen berbobot sama, tidak peduli
    berapa banyak field yang dikandungnya.

    Args:
        model: model yang dievaluasi.
        data: split dataset, umumnya `test_data`.
        device: device inferensi.
        max_samples: batas jumlah sample. Dibatasi demi waktu evaluasi, karena
            10 konfigurasi model harus dievaluasi berurutan.

    Returns:
        dict berisi rata-rata `Precision`, `Recall`, `F1`, dan `N-TED`.
    """
    precision_scores = []
    recall_scores = []
    f1_scores = []
    nted_scores = []

    total_samples = min(len(data), max_samples)

    for i in range(total_samples):
        sample = data[i]

        true_json = json.loads(sample["ground_truth"])["gt_parse"]

        pred_json, _ = predict_json(model, sample["image"], device=device)

        precision, recall, f1 = field_level_f1(pred_json, true_json)

        nted = normalized_tree_edit_distance(pred_json, true_json)

        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        nted_scores.append(nted)

    return {
        "Precision": sum(precision_scores) / len(precision_scores),
        "Recall": sum(recall_scores) / len(recall_scores),
        "F1": sum(f1_scores) / len(f1_scores),
        "N-TED": sum(nted_scores) / len(nted_scores),
    }

In [ ]:
def get_model_size_mb(model):
    """
    Mengukur ukuran model dari besar file state_dict yang diserialisasi.

    Menghitung `numel * itemsize` tidak valid untuk model terkuantisasi, karena
    tensor INT8 menyimpan skala dan zero-point tambahan. Menyimpan state_dict ke
    file sementara memberi angka yang apple-to-apple antara model FP32 dan INT8,
    yaitu ukuran nyata yang harus di-deploy.

    File sementara selalu dihapus di blok `finally`, termasuk bila penyimpanan
    gagal di tengah jalan.

    Args:
        model: model yang diukur.

    Returns:
        float ukuran dalam MB.
    """
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp:

        temporary_path = tmp.name

    try:
        torch.save(model.state_dict(), temporary_path)

        size_mb = os.path.getsize(temporary_path) / (1024 * 1024)

    finally:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)

    return size_mb

In [ ]:
def measure_latency(model, data, device="cpu", num_samples=100):
    """
    Mengukur rata-rata waktu inferensi per dokumen dalam milidetik.

    Tahap warm-up sebanyak `WARMUP_SAMPLES` dijalankan lebih dulu dan hasilnya
    dibuang, karena eksekusi pertama menanggung biaya sekali-jalan berupa
    alokasi memori CUDA, kompilasi kernel, dan pengisian cache — biaya yang
    tidak mencerminkan latensi kondisi steady-state.

    Yang diukur adalah seluruh proses generate autoregresif sampai token EOS,
    bukan satu forward pass, sehingga angka inilah yang relevan untuk deployment.

    Perbandingan latensi hanya valid bila `device` sama antar model. Seluruh
    pengukuran dilakukan di CUDA (GPU), termasuk model terkuantisasi weight-only
    yang dijalankan pada kernel CUDA.

    Args:
        model: model yang diukur.
        data: split dataset sumber gambar.
        device: device pengukuran.
        num_samples: jumlah sample yang diukur setelah warm-up.

    Returns:
        float rata-rata latensi dalam milidetik per sample.
    """
    model.eval()
    model.to(device)
    print(next(model.parameters()).device)

    # Warm-Up
    warmup_samples = min(WARMUP_SAMPLES, len(data))

    for i in range(warmup_samples):
        image = data[i]["image"].convert("RGB")

        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
        decoder_input_ids = processor.tokenizer(
            "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        with torch.no_grad():
            _ = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=MAX_LENGTH,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
            )

    # Latency measurement
    times = []
    total_samples = min(len(data), num_samples)

    for i in range(total_samples):
        image = data[i]["image"].convert("RGB")

        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
        decoder_input_ids = processor.tokenizer(
            "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        torch.cuda.synchronize()
        start = time.time()

        with torch.no_grad():
            _ = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=MAX_LENGTH,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
            )

        torch.cuda.synchronize()
        end = time.time()
        times.append(end - start)

    return sum(times) / len(times) * 1000

In [ ]:
class DonutFlopsWrapper(nn.Module):
    """
    Pembungkus model Donut agar dapat ditelusuri `fvcore.FlopCountAnalysis`.

    `FlopCountAnalysis` menelusuri graf komputasi lewat satu pemanggilan forward
    dengan argumen positional dan mengharapkan keluaran berupa tensor. Sementara
    itu VisionEncoderDecoderModel menerima argumen keyword dan mengembalikan
    objek output HuggingFace. Wrapper ini menyederhanakan signature-nya menjadi
    `(pixel_values, decoder_input_ids) -> logits`.

    `use_cache=False` dipakai supaya yang terhitung adalah satu forward pass
    penuh (teacher forcing), bukan langkah decoding yang memakai cache — bentuk
    inilah yang sebanding antar model.
    """

    def __init__(self, model):
        """
        Args:
            model: VisionEncoderDecoderModel yang akan dihitung FLOPs-nya.
        """
        super().__init__()
        self.model = model

    def forward(self, pixel_values, decoder_input_ids):
        """
        Args:
            pixel_values: tensor gambar hasil DonutProcessor.
            decoder_input_ids: tensor id token input decoder.

        Returns:
            Tensor logits decoder.
        """
        outputs = self.model(
            pixel_values=pixel_values,
            decoder_input_ids=decoder_input_ids,
            use_cache=False,
        )
        return outputs.logits

In [ ]:
def estimate_flops_gflops(model):
    """
    Mengestimasi beban komputasi model dalam GFLOPs untuk satu dokumen.

    Berbeda dengan latensi yang bergantung pada kondisi hardware, FLOPs adalah
    ukuran kompleksitas yang independen device. Metrik ini dipakai untuk
    membuktikan bahwa pruning benar-benar memangkas komputasi, bukan sekadar
    tampak lebih cepat karena kondisi mesin saat pengukuran.

    Perhitungan dilakukan pada salinan model di CPU agar tidak mengganggu state
    maupun alokasi memori GPU model asli. `decoder_input_ids` diisi penuh
    sepanjang `MAX_LENGTH` supaya semua model diukur pada panjang sequence yang
    sama.

    `FlopCountAnalysis` gagal pada modul yang tidak dikenalinya, misalnya layer
    terkuantisasi. Kegagalan ditangkap dan fungsi mengembalikan None sehingga
    proses evaluasi tetap berjalan; untuk model terkuantisasi FLOPs diambil dari
    versi FP32-nya lewat parameter `flops_source_model` pada `evaluate_model`.

    Args:
        model: model yang dihitung.

    Returns:
        float GFLOPs, atau None bila perhitungan gagal.
    """
    model_copy = copy.deepcopy(model)
    model_copy.eval()
    model_copy.to("cpu")

    sample = test_data[0]
    image = sample["image"].convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values

    decoder_input_ids = torch.full(
        size=(1, MAX_LENGTH),
        fill_value=processor.tokenizer.pad_token_id,
        dtype=torch.long,
    )

    decoder_input_ids[:, 0] = model_copy.config.decoder_start_token_id

    wrapper = DonutFlopsWrapper(model_copy)

    try:
        flops = FlopCountAnalysis(wrapper, (pixel_values, decoder_input_ids))
        flops.unsupported_ops_warnings(False)
        flops.uncalled_modules_warnings(False)
        return flops.total() / 1e9

    except Exception as e:
        print("FLOPs gagal dihitung:", e)
        return None

In [ ]:
def evaluate_model(model_name, model, flops_source_model=None, device=None):
    """
    Menjalankan seluruh pengukuran untuk satu konfigurasi model.

    Titik masuk tunggal evaluasi: menggabungkan metrik kualitas (Precision,
    Recall, F1, N-TED) dan metrik efisiensi (ukuran, latensi, FLOPs) menjadi
    satu baris hasil dengan kunci yang konsisten, sehingga seluruh konfigurasi
    dapat disusun menjadi satu tabel perbandingan di Bagian 9.

    Args:
        model_name: label model pada tabel hasil dan MLflow, mis. "DONUT-P50-KD".
        model: model yang dievaluasi.
        flops_source_model: model alternatif sebagai sumber perhitungan FLOPs.
            Dipakai untuk model terkuantisasi yang tidak dapat ditelusuri fvcore,
            diisi dengan versi FP32-nya.
        device: device evaluasi; default `EVAL_DEVICE`. Model terkuantisasi weight-only
            dapat dievaluasi di CUDA maupun CPU.

    Returns:
        dict satu baris hasil evaluasi.
    """
    if device is None:
        device = EVAL_DEVICE

    print(f"\nEvaluasi: {model_name} (device: {device})")

    extraction_metrics = evaluate_extraction(
        model, test_data, device=device, max_samples=EVAL_SAMPLES
    )

    latency = measure_latency(
        model, test_data, device=device, num_samples=LATENCY_SAMPLES
    )

    size_mb = get_model_size_mb(model)

    if flops_source_model is None:
        flops_source_model = model

    flops = estimate_flops_gflops(flops_source_model)

    result = {
        "Model": model_name,
        "Precision": extraction_metrics["Precision"],
        "Recall": extraction_metrics["Recall"],
        "F1-Score": extraction_metrics["F1"],
        "N-TED": extraction_metrics["N-TED"],
        "Size (MB)": size_mb,
        "Latency (ms/sample)": latency,
        "FLOPs (GFLOPs)": flops,
    }

    return result

In [ ]:
def add_baseline_comparison(result, baseline_result):
    """
    Menambahkan kolom selisih terhadap baseline pada satu baris hasil evaluasi.

    Konvensi tanda: kolom *Reduction* positif berarti lebih kecil/lebih cepat dari
    baseline, sedangkan `F1 Drop` dan `N-TED Increase` positif berarti kualitasnya
    menurun. FLOPs hanya dihitung bila nilai baseline tersedia dan bukan nol.

    Args:
        result: baris hasil dari `evaluate_model`.
        baseline_result: baris hasil model baseline.

    Returns:
        dict salinan `result` beserta kolom perbandingan.
    """
    compared_result = dict(result)

    compared_result["Size Reduction (%)"] = (
        (baseline_result["Size (MB)"] - result["Size (MB)"])
        / baseline_result["Size (MB)"]
        * 100
    )

    compared_result["Latency Reduction (%)"] = (
        (baseline_result["Latency (ms/sample)"] - result["Latency (ms/sample)"])
        / baseline_result["Latency (ms/sample)"]
        * 100
    )

    compared_result["F1 Drop"] = baseline_result["F1-Score"] - result["F1-Score"]

    compared_result["N-TED Increase"] = result["N-TED"] - baseline_result["N-TED"]

    baseline_flops = baseline_result.get("FLOPs (GFLOPs)")

    model_flops = result.get("FLOPs (GFLOPs)")

    if baseline_flops not in (None, 0) and model_flops is not None:
        compared_result["FLOPs Reduction (%)"] = (
            (baseline_flops - model_flops) / baseline_flops * 100
        )

    return compared_result

## 4 Weight-Only Quantization (PTQ) — Decoder INT8

In [ ]:
def apply_weight_only_quantization_decoder(model):
  """
  Mengubah layer pada decoder menjadi INT8 weight-only menggunakan torchao.

  Model disalin lebih dulu agar versi FP32-nya tetap utuh dan masih dibutuhkan sebagai
  sumber perhitungan FLOPs. Hanya bobot decoder yang dikuantisasi ke INT8 dengan
  konfigurasi Int8WeightOnlyConfig; nilai aktivasi tetap di FP32 dan bobot
  didekuantisasi kembali saat komputasi, sehingga penghematan utamanya terjadi
  pada memori; efek terhadap kecepatan inferensi bergantung pada dukungan kernel INT8
  perangkat keras. Pada GPU dengan kernel CUDA native, selisih latensi sebelum dan
  sesudah PTQ tetap diukur untuk menangkap potensi percepatan tersebut (Bagian 8).
  Model dipindahkan ke GPU sebelum kuantisasi.

  Args:
      model: model hasil KD yang akan dikuantisasi.

  Returns:
      salinan model dengan decoder terkuantisasi, berada di CUDA dan mode `eval()`.
  """
  quantized_model = copy.deepcopy(model)
  quantized_model.eval()
  quantized_model.to("cuda")

  quantize_(quantized_model.decoder, Int8WeightOnlyConfig())

  return quantized_model

In [ ]:
def check_quantization(model):
  """
  Menghitung jumlah layer Linear pada decoder yang benar-benar terkuantisasi torchao.

  Verifikasi ini perlu karena `quantize_` tidak melempar error bila kernel INT8 tidak
  tersedia atau layer tidak memenuhi syarat — hasil 0 dari total > 0 berarti kuantisasi
  gagal diam-diam. Pemeriksaan dilakukan dengan mengintrospeksi tipe data weight
  (apakah berasal dari modul `torchao`), bukan hanya mengandalkan struktur layer.

  Args:
      model: model hasil `apply_weight_only_quantization_decoder`.

  Returns:
      int jumlah layer Linear pada decoder yang weight-nya bertipe torchao.
  """

  total_linear = 0
  quantized_linear = 0

  for name, module in model.decoder.named_modules():
    if isinstance(module, nn.Linear):
      total_linear += 1

      weight = module.weight
      weight_type = type(weight)
      type_name = f"{weight_type.__module__}.{weight_type.__name__}"

      is_torchao = weight_type.__module__.startswith("torchao")

      if is_torchao:
        quantized_linear += 1

      print(
          f"{name:60s} "
          f"weight = {type_name} "
          f"device = {weight.device}"
      )

  print(f"\nLinear layers : {total_linear}")
  print(f"TorchAO quantized layers : {quantized_linear}")

  return quantized_linear


In [ ]:
# Evaluasi baseline (FP32) dengan metrik identik thesis_v3
baseline_result = evaluate_model("DONUT-Base", baseline_model)

# PTQ decoder-only (INT8 weight-only)
quantized_model = apply_weight_only_quantization_decoder(baseline_model)
quantized_layer_count = check_quantization(quantized_model)

# FLOPs model terkuantisasi diambil dari versi FP32-nya (PTQ tidak mengubah FLOPs)
quantized_result = evaluate_model(
    "DONUT-Base-Q",
    quantized_model,
    flops_source_model=baseline_model,
    device=EVAL_DEVICE,
)

# Kolom khusus PTQ: reduksi sebelum/sesudah kuantisasi terhadap baseline FP32
baseline_size = baseline_result["Size (MB)"]
baseline_latency = baseline_result["Latency (ms/sample)"]

quantized_result["PTQ Size Reduction (%)"] = (
    (baseline_size - quantized_result["Size (MB)"]) / baseline_size
) * 100

quantized_result["PTQ Latency Reduction (%)"] = (
    (baseline_latency - quantized_result["Latency (ms/sample)"]) / baseline_latency
) * 100

print("\nQuantized Linear Layers:", quantized_layer_count)


## 5 Logging ke Databricks (MLflow)

In [ ]:
def mlflow_metrics(metrics):
    """
    Menyaring dict metrik agar aman dikirim ke `mlflow.log_metrics`.

    MLflow hanya menerima float berhingga; nilai `None`, non-numerik, `NaN`, dan `inf`
    dibuang supaya satu metrik bermasalah tidak menggagalkan pencatatan seluruh run.

    Args:
        metrics: dict metrik mentah.

    Returns:
        dict berisi float berhingga saja.
    """
    cleaned = {}

    for key, value in metrics.items():
        if value is None:
            continue

        try:
            value = float(value)
        except (TypeError, ValueError):
            continue

        if np.isfinite(value):
            cleaned[key] = value

    return cleaned

In [ ]:
def get_environment_information():
    """
    Mengumpulkan versi library dan spesifikasi perangkat saat eksperimen dijalankan.

    Disimpan sebagai artifact tiap run agar angka latensi punya konteks hardware dan
    hasil dapat ditelusuri ulang.

    Returns:
        dict informasi environment.
    """
    return {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "mlflow_version": mlflow.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "train_device": TRAIN_DEVICE,
        "eval_device": EVAL_DEVICE,
        "gpu_name": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
        "cpu_threads": torch.get_num_threads(),
        "seed": SEED,
    }

In [ ]:
def safe_model_name(model_name):
    """
    Mengubah nama model menjadi bentuk yang aman dipakai sebagai nama file/folder.

    Args:
        model_name: label model, mis. "DONUT-P50-KD".

    Returns:
        str tanpa karakter selain huruf, angka, `_`, `.`, dan `-`.
    """
    return re.sub(f"[^A-Za-z0-9_.-]+", "_", model_name).strip("_")

In [ ]:
def save_model_to_mlflow(model, model_name, is_quantized=False):
    """
    Menyimpan bobot, processor, dan metadata model sebagai artifact MLflow.

    Format penyimpanan dibedakan: model terkuantisasi disimpan sebagai `state_dict` lewat `torch.save`
    (.pt) karena tensor INT8 hasil torchao tidak dapat dimuat ulang ke arsitektur FP32
    melalui `from_pretrained`, sedangkan
    model biasa disimpan sebagai safetensors lewat `save_pretrained`.

    `metadata.json` berisi `model_name` dan `is_quantized` — dua field inilah yang
    dibaca backend aplikasi untuk menentukan cara memuat model saat serving.

    Model dipindahkan ke CPU sebelum disimpan, lalu dikembalikan ke device asalnya
    kecuali bila terkuantisasi.

    Args:
        model: model yang disimpan.
        model_name: label model, dipakai sebagai nama folder artifact.
        is_quantized: menentukan format penyimpanan.

    Returns:
        None — artifact langsung dikirim ke run MLflow yang sedang aktif.
    """

    file_name = safe_model_name(model_name)

    try:
        original_device = next(model.parameters()).device

    except StopIteration:
        original_device = torch.device("cpu")

    model.to("cpu")
    model.eval()

    with tempfile.TemporaryDirectory() as temp_dir:
        artifact_root = Path(temp_dir) / file_name
        artifact_root.mkdir(parents=True, exist_ok=True)

        processor.save_pretrained(artifact_root / "processor")

        metadata = {
            "model_name": model_name,
            "base_model": MODEL_NAME,
            "framework": "pytorch",
            "is_quantized": is_quantized,
            "quantization": ("int8_weight_only_decoder_linear" if is_quantized else "none"),
            "quantized_module": ("decoder" if is_quantized else "none"),
            "seed": SEED,
        }

        with open(artifact_root / "metadata.json", "w", encoding="utf-8") as file:

            json.dump(metadata, file, indent=2, ensure_ascii=False)
        if is_quantized:
            model.config.to_json_file(artifact_root / "config.json")

            # torch.save(model, artifact_root / f"{file_name}.pt")
            torch.save(model.state_dict(), artifact_root / f"{file_name}.pt")

        else:
            model.save_pretrained(
                artifact_root / "huggingface_model", safe_serialization=True
            )

        mlflow.log_artifacts(str(artifact_root), artifact_path="model")

    if (not is_quantized) and (original_device.type != "cpu"):
        model.to(original_device)

In [ ]:
def log_model_to_mlflow(
    model_name,
    model,
    result,
    technique,
    is_quantized=False,
    baseline_result=None,
):
    """
    Mencatat satu konfigurasi model sebagai satu run MLflow yang lengkap.

    Meniru `log_experiment_to_mlflow` pada thesis_v3: menambahkan kolom
    perbandingan terhadap baseline bila `baseline_result` diberikan, mencatat
    parameter, metrik (mapping nama identik dengan v3), tag, artifact laporan
    (`result.json`, `environment.json`, `requirements.txt`), dan bobot model.

    Args:
        model_name: label model, dipakai sebagai `run_name`.
        model: model yang disimpan sebagai artifact.
        result: baris hasil dari `evaluate_model`.
        technique: deskripsi teknik kompresi.
        is_quantized: menentukan format penyimpanan model.
        baseline_result: baris hasil baseline untuk menghitung kolom perbandingan.

    Returns:
        dict `result` yang sudah dilengkapi `MLflow Run ID` dan `MLflow Artifact URI`.
    """

    if baseline_result is not None:
        result = add_baseline_comparison(result, baseline_result)
    else:
        result = dict(result)

    params = {
        "base_model": MODEL_NAME,
        "dataset": DATASET_NAME,
        "technique": technique,
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "train_samples": len(train_data),
        "validation_samples": len(val_data),
        "test_samples": len(test_data),
        "evaluation_samples": EVAL_SAMPLES,
        "latency_samples": LATENCY_SAMPLES,
        "warmup_samples": WARMUP_SAMPLES,
        "train_device": TRAIN_DEVICE,
        "eval_device": EVAL_DEVICE,
        "cpu_threads": CPU_NUM_THREADS,
        "quantization": ("int8_weight_only_decoder_linear" if is_quantized else "none"),
    }

    metric_mapping = {
        "Precision": "field_precision",
        "Recall": "field_recall",
        "F1-Score": "field_f1_score",
        "N-TED": "n_ted",
        "Size (MB)": "model_size_mb",
        "Latency (ms/sample)": "latency_ms_per_sample",
        "FLOPs (GFLOPs)": "estimated_gflops",
        "Size Reduction (%)": "size_reduction_pct",
        "Latency Reduction (%)": "latency_reduction_pct",
        "FLOPs Reduction (%)": "flops_reduction_pct",
        "F1 Drop": "f1_drop",
        "N-TED Increase": "n_ted_increase",
        "PTQ Size Reduction (%)": "ptq_size_reduction_pct",
        "PTQ Latency Reduction (%)": "ptq_latency_reduction_pct",
    }

    metrics = {
        mlflow_metric_name: result.get(result_name)
        for result_name, mlflow_metric_name in metric_mapping.items()
    }

    with mlflow.start_run(run_name=model_name) as run:

        mlflow.log_params(params)
        mlflow.log_metrics(mlflow_metrics(metrics))

        mlflow.set_tags(
            {
                "experiment_group": EXPERIMENT_GROUP,
                "architecture": "DONUT",
                "framework": "PyTorch",
                "task": "Document Information Extraction",
                "dataset": "CORD-v2",
                "optimization": technique,
                "model_format": (
                    "pytorch_pt" if is_quantized else "huggingface_safetensors"
                ),
                "mlflow.note.content": (
                    "Evaluasi menggunakan official test split CORD-v2. "
                    "Latency diukur pada evaluation device yang dicatat "
                    "dalam parameter dan environment artifact."
                ),
            }
        )

        with tempfile.TemporaryDirectory() as temp_dir:
            report_directory = Path(temp_dir)

            with open(report_directory / "result.json", "w", encoding="utf-8") as file:
                json.dump(result, file, indent=2, ensure_ascii=False)

            with open(report_directory / "environment.json", "w", encoding="utf-8") as file:
                json.dump(
                    get_environment_information(), file, indent=2, ensure_ascii=False
                )

            try:
                package_versions = subprocess.check_output(
                    [sys.executable, "-m", "pip", "freeze"], text=True
                )
                (report_directory / "requirements.txt").write_text(
                    package_versions, encoding="utf-8"
                )
            except Exception as e:
                (report_directory / "requirements_error.txt").write_text(
                    str(e), encoding="utf-8"
                )

            mlflow.log_artifacts(str(report_directory), artifact_path="report")

        save_model_to_mlflow(
            model=model, model_name=model_name, is_quantized=is_quantized
        )

        run_id = run.info.run_id
        experiment_id = run.info.experiment_id
        artifact_uri = mlflow.get_artifact_uri()

    result["MLflow Run ID"] = run_id
    result["MLflow Artifact URI"] = artifact_uri

    print("\n Berhasil dicatat ke MLflow")
    print("Model  :", model_name)
    print("Run ID :", run_id)
    print(
        "Databricks URL:",
        f"{os.environ['DATABRICKS_HOST']}"
        f"/#mlflow/experiments/"
        f"{experiment_id}/runs/{run_id}",
    )

    return result


In [ ]:
# Baseline dicatat paling awal karena hasilnya menjadi acuan pembanding.
baseline_result = log_model_to_mlflow(
    model_name="DONUT-Base",
    model=baseline_model,
    result=baseline_result,
    technique="baseline",
    is_quantized=False,
    baseline_result=None,
)

results = [baseline_result]


In [ ]:
quantized_result = log_model_to_mlflow(
    model_name="DONUT-Base-Q",
    model=quantized_model,
    result=quantized_result,
    technique="weight_only_int8_quantization",
    is_quantized=True,
    baseline_result=baseline_result,
)

results.append(quantized_result)


## 6 Hasil & Analisis

In [ ]:
df_results = pd.DataFrame(results)

baseline_size = df_results.loc[df_results["Model"] == "DONUT-Base", "Size (MB)"].values[0]
baseline_latency = df_results.loc[
    df_results["Model"] == "DONUT-Base", "Latency (ms/sample)"
].values[0]
baseline_flops = df_results.loc[
    df_results["Model"] == "DONUT-Base", "FLOPs (GFLOPs)"
].values[0]
baseline_f1 = df_results.loc[df_results["Model"] == "DONUT-Base", "F1-Score"].values[0]
baseline_nted = df_results.loc[df_results["Model"] == "DONUT-Base", "N-TED"].values[0]

df_results["Size Reduction (%)"] = (
    (baseline_size - df_results["Size (MB)"]) / baseline_size
) * 100

df_results["Latency Reduction (%)"] = (
    (baseline_latency - df_results["Latency (ms/sample)"]) / baseline_latency
) * 100

df_results["FLOPs Reduction (%)"] = (
    (baseline_flops - df_results["FLOPs (GFLOPs)"]) / baseline_flops
) * 100

df_results["F1 Drop"] = baseline_f1 - df_results["F1-Score"]
df_results["N-TED Increase"] = df_results["N-TED"] - baseline_nted

# Susun ulang kolom agar urutannya identik dengan tabel akhir thesis_v3
canonical_order = [
    "Model",
    "Precision",
    "Recall",
    "F1-Score",
    "N-TED",
    "Size (MB)",
    "Latency (ms/sample)",
    "FLOPs (GFLOPs)",
    "MLflow Run ID",
    "MLflow Artifact URI",
    "Size Reduction (%)",
    "Latency Reduction (%)",
    "F1 Drop",
    "N-TED Increase",
    "FLOPs Reduction (%)",
    "PTQ Size Reduction (%)",
    "PTQ Latency Reduction (%)",
]
df_results = df_results[canonical_order]

df_results


In [ ]:
def log_final_experiment_table(df_results):

    with mlflow.start_run(run_name="DONUT-BASE-QUANT-SUMMARY") as run:

        mlflow.set_tags(
            {
                "experiment_group": EXPERIMENT_GROUP,
                "run_type": "final_comparison",
                "architecture": "DONUT",
                "dataset": "CORD-v2",
            }
        )

        mlflow.log_params(
            {
                "total_model_configurations": len(df_results),
                "seed": SEED,
                "base_model": MODEL_NAME,
                "dataset": DATASET_NAME,
            }
        )

        with tempfile.TemporaryDirectory() as temp_dir:

            csv_path = Path(temp_dir) / "hasil_eksperimen_donut_quantisasi.csv"
            json_path = Path(temp_dir) / "hasil_eksperimen_donut_quantisasi.json"

            df_results.to_csv(csv_path, index=False)
            df_results.to_json(json_path, orient="records", indent=2)

            mlflow.log_artifact(str(csv_path), artifact_path="comparison")
            mlflow.log_artifact(str(json_path), artifact_path="comparison")

        print("Final comparison Run ID:", run.info.run_id)


log_final_experiment_table(df_results)
